### Librerias

In [31]:
import pandas as pd

### Carga de datos

In [32]:
df = pd.read_csv('../../data/raw_data 16-03-2026.csv',index_col=0)


In [33]:
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 8000 entries, 0 to 7999
Data columns (total 35 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   apartment_id                 8000 non-null   int64  
 1   name                         7997 non-null   object 
 2   description                  7946 non-null   object 
 3   host_id                      8000 non-null   int64  
 4   neighbourhood_name           8000 non-null   object 
 5   neighbourhood_district       4861 non-null   object 
 6   room_type                    8000 non-null   object 
 7   accommodates                 8000 non-null   int64  
 8   bathrooms                    7957 non-null   float64
 9   bedrooms                     7961 non-null   float64
 10  beds                         7992 non-null   float64
 11  amenities_list               7983 non-null   object 
 12  price                        7829 non-null   float64
 13  minimum_nights         

In [ ]:
#Total de registros
print("Total filas:", len(df))
#Total de IDs únicos
print("IDs únicos:", df['apartment_id'].nunique())
#Total de registros duplicados 
print("Duplicados (estilo SQL):", len(df) - df['apartment_id'].nunique())

df['apartment_id'].isna().sum()

Total filas: 8000
IDs únicos: 7693
Duplicados (estilo SQL): 307


np.int64(0)

### Limpieza


Nulos

In [35]:
df.isnull().sum()

apartment_id                      0
name                              3
description                      54
host_id                           0
neighbourhood_name                0
neighbourhood_district         3139
room_type                         0
accommodates                      0
bathrooms                        43
bedrooms                         39
beds                              8
amenities_list                   17
price                           171
minimum_nights                    0
maximum_nights                    0
has_availability                550
availability_30                   0
availability_60                   0
availability_90                   0
availability_365                  0
number_of_reviews                 0
first_review_date              1614
last_review_date               1615
review_scores_rating           1696
review_scores_accuracy         1705
review_scores_cleanliness      1699
review_scores_checkin          1710
review_scores_communication 

Para el análisis de esta semana se ha aplicado la siguiente estrategia:

Eliminación en la variable price: Se han descartado las filas sin precio. Al ser la métrica principal del estudio y presentar un porcentaje de nulos mínimo (1.8%), su eliminación directa es la opción más segura para no introducir sesgos mediante imputaciones artificiales.

Conservación en las reseñas: Los valores nulos de las dos columnas de valoraciones se mantienen intactos. Estas ausencias no son errores, sino información de negocio válida (apartamentos nuevos o sin reservas previas), por lo que conservarlos refleja fielmente la realidad del mercado.

In [36]:
df=df.dropna(subset=['price'])

In [37]:
df[df['name'].isnull()]

,apartment_id,name,description,host_id,neighbourhood_name,neighbourhood_district,room_type,accommodates,bathrooms,bedrooms,...,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,is_instant_bookable,reviews_per_month,country,city,insert_date
1507,7164589,NaN,"exterior, bright, and charming room, in the ce...",37525983,Palacio,Centro,Private room,1,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,FALSO,NaN,spain,madrid,17/10/2020
1581,7576999,NaN,"This fantastic private bedroom, located in a p...",14415017,el Putxet i el Farr�,Sarri�-Sant Gervasi,Private room,2,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,FALSO,NaN,spain,barcelona,12/09/2020
2165,11687495,NaN,What everyone loves most about my place is the...,48387429,Simancas,San Blas - Canillejas,Entire home/apt,4,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,FALSO,NaN,spain,madrid,07/06/2019


In [38]:
df['description'] = df['description'].fillna('(sin contestar)')
df['neighbourhood_district'] = df['neighbourhood_district'].fillna('(sin contestar)')
df['amenities_list'] = df['amenities_list'].fillna('(sin contestar)')

Duplicados

Se ordena el dataset por identificador y fecha para conservar únicamente el registro más reciente en la tabla principal, archivando las versiones anteriores en un DataFrame secundario a modo de histórico.


In [39]:
df = df.sort_values(by=['apartment_id','insert_date'], ascending=[True, False])


dfanunciosantiguos = df[df.duplicated(subset=['apartment_id'],keep=False)].copy()

df = df.drop_duplicates(subset=['apartment_id'], keep='first')


In [40]:
print("Total filas:", len(df))
print("IDs únicos:", df['apartment_id'].nunique())
print("Duplicados (estilo SQL):", len(df) - df['apartment_id'].nunique())
df['apartment_id'].isna().sum()

Total filas: 7537
IDs únicos: 7537
Duplicados (estilo SQL): 0


np.int64(0)

In [41]:
es_unico = df['apartment_id'].is_unique
print(f"¿Son todos los IDs únicos?: {es_unico}")

¿Son todos los IDs únicos?: True


In [42]:
# Estandarización de texto: formato título para los nombres de las ciudades
df['city'] = df['city'].str.title()

### Transformación 

Para garantizar la integridad del análisis, se ha llevado a cabo un proceso de estandarización estructural del dataset. Esto ha incluido el casting de variables (conversión de las columnas a sus tipos de datos nativos correspondientes, como numéricos, booleanos o fechas) para permitir operaciones matemáticas correctas. Asimismo, se han corregido inconsistencias y anomalías de formato detectadas en varias columnas, asegurando que la información sea coherente y esté optimizada para la fase de modelado.

Se recalcula la columna de reviews_per_month ya que el equipo se ha dado cuenta de que los numeros no son correctos
Convierto las columnas de las reseñas a formato fecha para poder calcular el tiempo pasado desde 'first_review_date'(primera fecha de la que disponemos) hasta la 'insert_date'(ultima fecha de la que disponemos) para que el calculo sea mas cercano a la realidad

In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7537 entries, 0 to 7999
Data columns (total 35 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   apartment_id                 7537 non-null   int64  
 1   name                         7534 non-null   object 
 2   description                  7537 non-null   object 
 3   host_id                      7537 non-null   int64  
 4   neighbourhood_name           7537 non-null   object 
 5   neighbourhood_district       7537 non-null   object 
 6   room_type                    7537 non-null   object 
 7   accommodates                 7537 non-null   int64  
 8   bathrooms                    7494 non-null   float64
 9   bedrooms                     7499 non-null   float64
 10  beds                         7530 non-null   float64
 11  amenities_list               7537 non-null   object 
 12  price                        7537 non-null   float64
 13  minimum_nights         

In [44]:
df['bathrooms']=df['bathrooms'].astype('Int64')

df['beds']=df['beds'].astype('Int64')

df['bedrooms']=df['bedrooms'].astype('Int64')



In [45]:



df['first_review_date']= pd.to_datetime(df['first_review_date'], dayfirst=True)

df['last_review_date']= pd.to_datetime(df['last_review_date'], dayfirst=True)
df['insert_date'] = pd.to_datetime(df['insert_date'], dayfirst=True)

# Calculamos los meses transcurridos
meses = (df['insert_date'] - df['first_review_date']).dt.days / 30.4



#df['reviews_per_month'] = round(((df['number_of_reviews'] / meses)),2)



In [46]:


df['review_scores_rating']=df['review_scores_rating']/10
df['review_scores_rating']=df['review_scores_rating'].astype('Int64')

# Esta línea convierte lo que no sea 'VERDADERO' a False (Asumiendo que si esta nulo es porque no tiene disponibilidad)
df['has_availability'] = df['has_availability'] == 'VERDADERO'

df['is_instant_bookable'] = df['is_instant_bookable'] == 'VERDADERO'



In [47]:
# Resumen de rangos para todas las columnas de números
resumen_limpieza = df.select_dtypes(include='number').agg(['min', 'max'])
print(resumen_limpieza)

     apartment_id    host_id  accommodates  bathrooms  bedrooms  beds   price  \
min         11964      10704             1          0         0     0    60.0   
max      32423292  336524176            16         11        12    22  6071.0   

     minimum_nights  maximum_nights  availability_30  ...  availability_365  \
min               1               1                0  ...                 0   
max             365            1125               30  ...               365   

     number_of_reviews  review_scores_rating  review_scores_accuracy  \
min                  0                    20                    20.0   
max                588                   100                   100.0   

     review_scores_cleanliness  review_scores_checkin  \
min                       20.0                   20.0   
max                      100.0                  100.0   

     review_scores_communication  review_scores_location  review_scores_value  \
min                         20.0                

### Creación de variables

Se ha generado una nueva variable booleana para identificar los apartamentos con valoraciones superiores a 80

In [48]:
df['reviews 80+']=df['review_scores_rating'] >= 80 

In [49]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7537 entries, 0 to 7999
Data columns (total 36 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   apartment_id                 7537 non-null   int64         
 1   name                         7534 non-null   object        
 2   description                  7537 non-null   object        
 3   host_id                      7537 non-null   int64         
 4   neighbourhood_name           7537 non-null   object        
 5   neighbourhood_district       7537 non-null   object        
 6   room_type                    7537 non-null   object        
 7   accommodates                 7537 non-null   int64         
 8   bathrooms                    7494 non-null   Int64         
 9   bedrooms                     7499 non-null   Int64         
 10  beds                         7530 non-null   Int64         
 11  amenities_list               7537 non-null   obj

In [50]:
df

,apartment_id,name,description,host_id,neighbourhood_name,neighbourhood_district,room_type,accommodates,bathrooms,bedrooms,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,is_instant_bookable,reviews_per_month,country,city,insert_date,reviews 80+
0,11964,A ROOM WITH A VIEW,Private bedroom in our attic apartment. Right ...,45553,Centro,(sin contestar),Private room,2,2,1,...,100.0,100.0,100.0,100.0,False,75.0,spain,Malaga,2018-07-31,True
1,21853,Bright and airy room,We have a quiet and sunny room with a good vie...,83531,C�rmenes,Latina,Private room,1,1,1,...,100.0,100.0,80.0,90.0,False,52.0,spain,Madrid,2020-01-10,True
2,32347,Explore Cultural Sights from a Family-Friendly...,Open French doors and step onto a plant-filled...,139939,San Vicente,Casco Antiguo,Entire home/apt,4,1,2,...,100.0,100.0,100.0,100.0,True,142.0,spain,Sevilla,2019-07-29,True
3,35379,Double 02 CasanovaRooms Barcelona,Room at a my apartment. Kitchen and 2 bathroom...,152232,l'Antiga Esquerra de l'Eixample,Eixample,Private room,2,2,1,...,100.0,100.0,100.0,90.0,True,306.0,spain,Barcelona,2020-01-10,True
4,35801,Can Torras Farmhouse Studio Suite,Lay in bed & watch sunlight change the mood of...,153805,Quart,(sin contestar),Private room,5,1,2,...,100.0,100.0,100.0,100.0,False,39.0,spain,Girona,2019-02-19,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,32392193,Espectacular habitaci�n,(sin contestar),238089984,Sant Antoni,Eixample,Private room,1,1,1,...,NaN,NaN,NaN,NaN,True,NaN,spain,Barcelona,2019-10-16,<NA>
7996,32392774,? Tu Hogar de Lujo ????? en el Centro de Sevilla,"Exclusivo, amplio y luminoso alojamiento situa...",243246681,Arenal,Casco Antiguo,Entire home/apt,6,2,3,...,100.0,100.0,100.0,100.0,False,157.0,spain,Sevilla,2021-01-31,True
7997,32395123,Rooms by G Bella Mar�a 3,The 2-star Bella Maria has 24-hourreception an...,159933359,Felanitx,(sin contestar),Entire home/apt,2,1,1,...,NaN,NaN,NaN,NaN,True,NaN,spain,Mallorca,2019-04-24,<NA>
7998,32407332,LUMINOSO Y ENCANTADOR PISO CERCA DE TODO,PISO MUY ILUMINADO CON UNA TERRAZA ESTUPENDA C...,187631805,Proven�als del Poblenou,Sant Mart�,Private room,3,2,2,...,100.0,100.0,100.0,100.0,True,389.0,spain,Barcelona,2019-08-12,True


Se genera una archivo CSV con los datos limpios y otro con los anuncios antiguos

In [51]:
# Generación del CSV bloqueada (el archivo ya se encuentra en el directorio del proyecto).
#df.to_csv('../../data/clean_data 16-03-2026.csv', index=False)

#dfanunciosantiguos.to_csv('../../data/Anuncios antiguos_09-03-2026.csv', index=False)